# 🚀 Единый пайплайн разработки и эксплуатации робота PRL/USDT
Актуальная 7-этапная последовательность работы с проектом:
1. **Мониторинг VPS** (`5` коллекторов 24/7)
2. **Синхронизация данных** (Google Drive + горячий хвост VPS + боевая телеметрия)
3. **Сборка 1s-признаков и разметки** (`642` признака + 4 горизонта `H = 5:3, 12:3.5, 25:3, 60:3`)
4. **Планировщик окон Walk-Forward** (контроль накопления новых окон для Учителя и Учеников)
5. **Обучение роя предсказателей** (`LightGBM` + экспорт `.npz` для 4 нейросетевых архитектур на GPU)
6. **Обучение RL-агента исполнения** (`Maskable PPO v3` + GPU Population Search на 10 уровнях L2 FIFO)
7. **Шлюз безоткатной эволюции и боевой демон** (`Non-Regression Gate` + `LiveTriadRLTrader`)

In [ ]:
# ЭТАП 1. Проверка статуса 5 коллекторов на VPS (81.19.137.18)
!.venv/bin/python -m _tools.monitor_vps --once

In [ ]:
# ЭТАП 2. Синхронизация архивов Google Drive + горячих CSV и телеметрии с VPS
!.venv/bin/python -m _tools.sync_all_data
!.venv/bin/python -m _tools.check_realtime_data

In [ ]:
# ЭТАП 3. Сборка 1-секундной сетки (642 признака) и 7-классовой разметки (H=5:3, 12:3.5, 25:3, 60:3)
!.venv/bin/python -m _tools.prepare_1s_dataset --coin prl
!.venv/bin/python -m _tools.data_labeler --coin prl --pairs "5:3,12:3.5,25:3,60:3"

In [ ]:
# ЭТАП 4. Калькулятор новых окон Walk-Forward (Учитель LightGBM + Ученики Нейросети)
!.venv/bin/python -m _tools.plan_rolling_windows --coin prl --horizons 5,12,25,60 --min-test-sec 21600

In [ ]:
# ЭТАП 5. Дообучение окон роя (LightGBM + экспорт .npz и дообучение GPU-нейросетей)
!.venv/bin/python _tools/train_rolling_swarm.py --coins prl --horizons 5,12,25,60
!.venv/bin/python _tools/export_colab_dataset.py

In [ ]:
# ЭТАП 6. Инкрементальное обновление кэша полигона (176ч) и обучение RL-агента исполнения (Maskable PPO v3)
!.venv/bin/python _tools/train_rl_execution.py --coin prl --tail-hours 176 --epochs 30 --steps 32768 --pop-size 8

In [ ]:
# ЭТАП 7. Проверка на шлюзе безоткатной эволюции (Non-Regression Gate) и аудит OOS
!.venv/bin/python _tools/regression_gate.py